In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = "cinedata_medallion"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"
bronze_volume = f"/Volumes/{catalog}/bronze/landing"


In [0]:
def normalizar_texto(df, colunas=None):
    if colunas is None:
        colunas = [c for c, tipo in df.dtypes if tipo == "string"]

    for c in colunas:
        c_tratada = F.lower(F.trim(F.col(c)))
        c_tratada = F.regexp_replace(c_tratada, r"[^\p{L}\p{N}\s-]", "")   # ruídos: tudo que não é letra, número, espaço ou hífen
        c_tratada = F.regexp_replace(c_tratada, r"-", " ")                 # hífens viram espaço
        c_tratada = F.regexp_replace(c_tratada, r"\s+", " ")               # espaços múltiplos viram um só
        c_tratada = F.trim(c_tratada)
        df = df.withColumn(c, c_tratada)

    return df

In [0]:
#silver.tb_info_filmes (origem: tb_movies_info)

#ler do bronze
df_bronze_info_filmes = spark.table(f"{bronze_schema}.movies_info")
print(f"colunas de df: {df_bronze_info_filmes.columns}")


# tradução do status (chaves já normalizadas pela normalizar_texto)
traducao_status = {
    "released":        "Lançado",
    "post production": "Pós-Produção",
    "in production":   "Em Produção",
    "planned":         "Planejado",
    "rumored":         "Rumores",
    "canceled":        "Cancelado",
}
mapa_status = F.create_map(*[F.lit(x) for par in traducao_status.items() for x in par])

# janela de deduplicação: versão mais recente de cada filme
janela_recente = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())

df_silver_info_filmes = (
    df_bronze_info_filmes
    # deduplicação
    .withColumn("rn", F.row_number().over(janela_recente))
    .filter(F.col("rn") == 1)
    .drop("rn")

    # status: normaliza -> traduz -> não mapeado vira "Não Informado"
    .transform(lambda d: normalizar_texto(d, ["status"]))
    .withColumn("status", F.coalesce(mapa_status[F.col("status")], F.lit("Não Informado")))

    # data multi-formato: testa cada padrão, o primeiro que converter vence
    .withColumn("release_date", F.coalesce(
        F.to_date(F.expr("try_to_timestamp(release_date, 'yyyy-MM-dd')")),
        F.to_date(F.expr("try_to_timestamp(release_date, 'dd/MM/yyyy')")),
        F.to_date(F.expr("try_to_timestamp(release_date, 'MM-dd-yyyy')")),
    ))

    # tipos
    .withColumn("id", F.expr("try_cast(id AS INT)"))
    .withColumn("runtime", F.expr("try_cast(runtime AS INT)"))
    .filter(F.col("id").isNotNull())

    # renomear conforme mapeamento
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("title", "titulo")
    .withColumnRenamed("original_title", "titulo_original")
    .withColumnRenamed("release_date", "data_lancamento")
    .withColumnRenamed("runtime", "duracao_minutos")
    .withColumnRenamed("original_language", "idioma_original")
    .withColumnRenamed("status", "status_filme")
    .withColumnRenamed("overview", "sinopse")
    .withColumnRenamed("tagline", "frase_divulgacao")

    # coluna derivada
    .withColumn("ano_lancamento", F.year("data_lancamento"))

    # colunas finais (tconst e ingestion_datetime ficam de fora)
    .select("id_filme", "titulo", "titulo_original", "data_lancamento", "ano_lancamento",
            "duracao_minutos", "idioma_original", "status_filme", "sinopse", "frase_divulgacao")
)

# gravar
(df_silver_info_filmes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{silver_schema}.tb_info_filmes"))

print(f"{silver_schema}.tb_info_filmes: {df_silver_info_filmes.count()} linhas")

display(df_silver_info_filmes.orderBy(F.col("id_filme")))

In [0]:

# ler do bronze
df_bronze_financeiro = spark.table(f"{bronze_schema}.movies_financials")
print(f"colunas de df: {df_bronze_financeiro.columns}")

# cotação mais recente do dólar (bronze.tb_cotacao_dolar)
cotacao_dolar = (
    spark.table(f"{bronze_schema}.tb_cotacao_dolar")
    .orderBy(F.col("dataHoraCotacao").desc())
    .select("cotacaoCompra")
    .first()[0]
)
print(f"cotação utilizada: {cotacao_dolar}")
taxa_brl = F.lit(cotacao_dolar).cast("decimal(10,4)")

# textos que representam ausência de dado
textos_ausentes = ["UNKNOWN", "NÃO INFORMADO", "NAO INFORMADO", "N/A", "NA", "NULL", "NONE", " ",""]

def tratar_valor(coluna):
    txt = F.upper(F.trim(F.col(coluna)))
    txt = F.when(txt.isin(textos_ausentes), None).otherwise(txt)    # ausência -> NULL
    txt = F.regexp_replace(txt, r"R\$|USD|\$|,|\s", "")               # moeda, milhar, espaços

    multiplicador = (F.when(txt.endswith("K"), 1_000)
                      .when(txt.endswith("M"), 1_000_000)
                      .when(txt.endswith("B"), 1_000_000_000)
                      .otherwise(1))
    numero_txt = F.regexp_replace(txt, r"[KMB]$", "")

    valor = F.when(numero_txt.rlike(r"^-?\d+(\.\d+)?$"),               # só converte se for número válido
                   numero_txt.cast("decimal(18,2)") * multiplicador)
    return F.when(valor > 0, valor).cast("decimal(18,2)")             # zero/negativo -> NULL

# janela de deduplicação: mais recente; empate -> linha com mais informação
janela_recente = Window.partitionBy("id").orderBy(
    F.col("ingestion_datetime").desc(),
    F.col("budget").isNull().asc(),
    F.col("revenue").isNull().asc(),
)

df_silver_financeiro = (
    df_bronze_financeiro
    # higienização e conversão
    .withColumn("budget", tratar_valor("budget"))
    .withColumn("revenue", tratar_valor("revenue"))
    .withColumn("id", F.expr("try_cast(id AS INT)"))
    .filter(F.col("id").isNotNull())

    # deduplicação
    .withColumn("rn", F.row_number().over(janela_recente))
    .filter(F.col("rn") == 1)
    .drop("rn")

    # renomear conforme mapeamento
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("budget", "orcamento_usd")
    .withColumnRenamed("revenue", "receita_usd")

    # conversão para BRL
    .withColumn("orcamento_brl", (F.col("orcamento_usd") * taxa_brl).cast("decimal(18,2)"))
    .withColumn("receita_brl", (F.col("receita_usd") * taxa_brl).cast("decimal(18,2)"))

    # lucro: só existe se orçamento E receita existirem
    .withColumn("lucro_usd", (F.col("receita_usd") - F.col("orcamento_usd")).cast("decimal(18,2)"))
    .withColumn("lucro_brl", (F.col("receita_brl") - F.col("orcamento_brl")).cast("decimal(18,2)"))

    # margem %: protegida contra divisão por zero/nulo
    .withColumn("margem_lucro_pct",
        F.when(F.col("receita_usd") > 0,
               F.col("lucro_usd") / F.col("receita_usd") * 100).cast("decimal(10,2)"))

    # colunas finais
    .select("id_filme", "orcamento_usd", "receita_usd", "orcamento_brl", "receita_brl",
            "lucro_usd", "lucro_brl", "margem_lucro_pct")
)

# gravar
(df_silver_financeiro.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{silver_schema}.tb_financeiro_filmes"))

print(f"{silver_schema}.tb_financeiro_filmes: {df_silver_financeiro.count()} linhas")

display(df_silver_financeiro.orderBy(F.col("id_filme").desc()))

In [0]:
# ler do bronze
df_bronze_metricas = spark.table(f"{bronze_schema}.movies_metrics")
print(f"colunas de df: {df_bronze_metricas.columns}")

# --- funções de conversão segura ---
REGEX_DECIMAL = r"^-?\d+(\.\d+)?$"
REGEX_INTEIRO = r"^-?\d+$"

def para_decimal(coluna):
    """texto -> decimal; corrige vírgula decimal; texto inválido -> NULL"""
    txt = F.trim(F.col(coluna))
    txt = F.when(txt.contains(",") & txt.contains("."),               # 1.234,56 -> 1234,56
                 F.regexp_replace(txt, r"\.", "")).otherwise(txt)
    txt = F.regexp_replace(txt, ",", ".")                              # 89,985 -> 89.985
    return F.when(txt.rlike(REGEX_DECIMAL), txt.cast("decimal(18,3)"))

def para_inteiro(coluna):
    """texto -> bigint; só aceita inteiros (6.6 em contagem = coluna deslocada) -> NULL"""
    txt = F.trim(F.col(coluna))
    return F.when(txt.rlike(REGEX_INTEIRO), txt.cast("bigint"))

def nota_valida(valor):
    """nota fora de 0-10 (inclui erro de escala) -> NULL"""
    return F.when(valor.between(0, 10), valor).cast("decimal(5,3)")

def nao_negativo(valor):
    """negativo -> NULL"""
    return F.when(valor >= 0, valor)

# janela de deduplicação: mais recente; empate -> linha com mais campos válidos
qtd_preenchidos = sum(F.col(c).isNotNull().cast("int") for c in
                      ["popularity", "vote_average", "vote_count", "averageRating", "numVotes"])

janela_recente = Window.partitionBy("id").orderBy(
    F.col("ingestion_datetime").desc(),
    F.col("qtd_preenchidos").desc(),
)

df_silver_metricas = (
    df_bronze_metricas
    # conversão segura + regras de negócio
    .withColumn("id", F.expr("try_cast(id AS INT)"))
    .withColumn("popularity",    nao_negativo(para_decimal("popularity")).cast("decimal(10,3)"))
    .withColumn("vote_average",  nota_valida(para_decimal("vote_average")))
    .withColumn("averageRating", nota_valida(para_decimal("averageRating")))
    .withColumn("vote_count",    nao_negativo(para_inteiro("vote_count")))
    .withColumn("numVotes",      nao_negativo(para_inteiro("numVotes")))
    .filter(F.col("id").isNotNull())

    # deduplicação
    .withColumn("qtd_preenchidos", qtd_preenchidos)
    .withColumn("rn", F.row_number().over(janela_recente))
    .filter(F.col("rn") == 1)
    .drop("rn", "qtd_preenchidos")

    # renomear conforme mapeamento
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("popularity", "popularidade")
    .withColumnRenamed("vote_average", "nota_media_tmdb")
    .withColumnRenamed("vote_count", "qtd_votos_tmdb")
    .withColumnRenamed("averageRating", "nota_media_imdb")
    .withColumnRenamed("numVotes", "qtd_votos_imdb")

    # colunas finais
    .select("id_filme", "popularidade", "nota_media_tmdb", "qtd_votos_tmdb",
            "nota_media_imdb", "qtd_votos_imdb")
)

# gravar
(df_silver_metricas.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(f"{silver_schema}.tb_metricas_engajamento"))

print(f"{silver_schema}.tb_metricas_engajamento: {df_silver_metricas.count()} linhas")
display(df_silver_metricas.orderBy("id_filme"))

In [0]:
#ler do bronze
df_bronze_avaliacoes = spark.table(f"{bronze_schema}.movies_reviews")
print(f"colunas de df: {df_bronze_avaliacoes.columns}")

nota_txt = F.regexp_replace(F.trim(F.col("nota")), ",", ".")
nota_dec = F.when(nota_txt.rlike(r"^-?\d+(\.\d+)?$"), nota_txt.cast("decimal(18,2)"))

df_silver_avaliacoes = (
    df_bronze_avaliacoes
    .withColumn("id", F.expr("try_cast(id AS INT)"))
    .withColumn("nome", F.trim(F.col("nome")))
    .filter(F.col("id").isNotNull())

    # nota: fora de 0-10 -> NULL
    .withColumn("nota", F.when(nota_dec.between(0, 10), nota_dec).cast("decimal(4,2)"))

    # comentário: nulo, vazio ou só espaços -> "Sem comentário"
    .withColumn("comentario",
        F.when(F.trim(F.coalesce(F.col("comentario"), F.lit(""))) == "", F.lit("Sem comentário"))
         .otherwise(F.trim(F.col("comentario"))))

    # duplicidade integral (depois de padronizar, "" e "   " contam como iguais)
    .dropDuplicates(["id", "nome", "nota", "comentario"])

    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("nome", "nome_usuario")
    .withColumnRenamed("nota", "nota_usuario")
    .withColumnRenamed("comentario", "comentario_usuario")
    .select("id_filme", "nome_usuario", "nota_usuario", "comentario_usuario")
)

(df_silver_avaliacoes.write
    .format("delta").mode("overwrite").option("overwriteSchema", True)
    .saveAsTable(f"{silver_schema}.tb_avaliacoes_usuarios"))

print(f"{silver_schema}.tb_avaliacoes_usuarios: {df_silver_avaliacoes.count()} linhas")
display(df_silver_avaliacoes.orderBy("nome_usuario", "id_filme"))

In [0]:
#ler do bronze
df_bronze_credits = spark.table(f"{bronze_schema}.credits_and_tags")

# domínio de gêneros válidos (TMDB): chave normalizada -> nome padronizado
generos_validos = ["Action", "Adventure", "Animation", "Comedy", "Crime", "Documentary",
                   "Drama", "Family", "Fantasy", "History", "Horror", "Music", "Mystery",
                   "Romance", "Science Fiction", "TV Movie", "Thriller", "War", "Western"]
mapa_generos = F.create_map(*[F.lit(x) for g in generos_validos for x in (g.lower(), g)])

df_silver_generos = (
    df_bronze_credits
    .withColumn("id", F.expr("try_cast(id AS INT)"))
    .filter(F.col("id").isNotNull())

    # separadores inconsistentes (, ; |) -> split -> explode
    .withColumn("genero", F.explode(F.split(F.col("genres"), r"\s*[,;|]\s*")))

    # só fica o que pertence ao domínio; resíduos (números, frases, .jpg, vazios) viram NULL
    .withColumn("genero", mapa_generos[F.lower(F.trim(F.col("genero")))])
    .filter(F.col("genero").isNotNull())

    .withColumnRenamed("id", "id_filme")
    .select("id_filme", "genero")
    .dropDuplicates()
)

(df_silver_generos.write
    .format("delta").mode("overwrite").option("overwriteSchema", True)
    .saveAsTable(f"{silver_schema}.tb_generos"))

print(f"{silver_schema}.tb_generos: {df_silver_generos.count()} linhas")
display(df_silver_generos.orderBy("id_filme", "genero"))

In [0]:
#ler do bronze
df_bronze_credits = spark.table(f"{bronze_schema}.credits_and_tags")

def explodir_entidade(df, coluna, tipo):
    """separa a coluna pelos delimitadores e gera uma linha por entidade, com o tipo"""
    return df.select(
        F.expr("try_cast(id AS INT)").alias("id_filme"),
        F.explode(F.split(F.col(coluna), r"\s*[,;|]\s*")).alias("nome"),
        F.lit(tipo).alias("tipo_entidade"),
    )

# uma fatia por tipo de entidade
df_atores      = explodir_entidade(df_bronze_credits, "cast", "Ator")
df_diretores   = explodir_entidade(df_bronze_credits, "directors", "Diretor")
df_roteiristas = explodir_entidade(df_bronze_credits, "writers", "Roteirista")
df_produtoras  = explodir_entidade(df_bronze_credits, "production_companies", "Produtora")

# empilhar as quatro fatias
df_entidades = (
    df_atores
    .unionByName(df_diretores)
    .unionByName(df_roteiristas)
    .unionByName(df_produtoras)
)

nome = F.trim(F.col("nome"))
eh_residuo = (
    (nome == "")
    | nome.rlike(r"^[\d\s.,/:\-]+$")                  # numérico deslocado (0.6, 774149, 10/09)
    | nome.rlike(r"(?i)\.(jpg|png)$")                 # caminho de imagem
    | nome.rlike(r"[\\\"]")                           # restos de aspas/escape de sinopse
    | (F.size(F.split(nome, r"\s+")) > 8)             # frase descritiva, não nome
)

df_silver_pessoas = (
    df_entidades
    .filter(F.col("id_filme").isNotNull())
    .withColumn("nome", nome)
    .filter(~eh_residuo)
    .withColumn("nome", F.initcap(F.lower(F.col("nome"))))   # capitalização padronizada
    .select("id_filme", "nome", "tipo_entidade")
    .dropDuplicates()
)

(df_silver_pessoas.write
    .format("delta").mode("overwrite").option("overwriteSchema", True)
    .saveAsTable(f"{silver_schema}.tb_pessoas_empresas"))

print(f"{silver_schema}.tb_pessoas_empresas: {df_silver_pessoas.count()} linhas")
display(df_silver_pessoas.groupBy("tipo_entidade").count())
display(df_silver_pessoas.orderBy("id_filme", "nome", "tipo_entidade"))

In [0]:
#ler cotacao dolar do bronze
df_bronze_cotacao = spark.table(f"{bronze_schema}.tb_cotacao_dolar")
print(f"colunas de df: {df_bronze_cotacao.columns}")


display(df_bronze_cotacao.orderBy("dataHoraCotacao"))


# 1. uma cotação por dia (a mais recente, caso a API tenha sido chamada mais de uma vez)
janela_dia = Window.partitionBy("data_cotacao").orderBy(
    F.col("dataHoraCotacao").desc(),
    F.col("ingestion_datetime").desc(),
)

df_cotacoes_dia = (
    df_bronze_cotacao
    .withColumn("data_cotacao", F.to_date(F.substring("dataHoraCotacao", 1, 10)))
    .withColumn("cotacao_compra", F.col("cotacaoCompra").cast("decimal(10,4)"))
    .filter(F.col("data_cotacao").isNotNull() & F.col("cotacao_compra").isNotNull())
    .withColumn("rn", F.row_number().over(janela_dia))
    .filter(F.col("rn") == 1)
    .select("data_cotacao", "cotacao_compra")
)

# 2. calendário contínuo: do primeiro dia com cotação até hoje
df_calendario = (
    df_cotacoes_dia
    .agg(F.min("data_cotacao").alias("inicio"))
    .select(F.explode(F.sequence(F.col("inicio"), F.current_date())).alias("data_cotacao"))
)

# 3. left join + forward fill
janela_ffill = (Window.orderBy("data_cotacao")
                .rowsBetween(Window.unboundedPreceding, Window.currentRow))

df_silver_cotacao = (
    df_calendario
    .join(df_cotacoes_dia, on="data_cotacao", how="left")
    .withColumn("sem_cotacao", F.col("cotacao_compra").isNull())    # marca os dias sem cotação
    .withColumn("cotacao_compra", F.last("cotacao_compra", ignorenulls=True).over(janela_ffill))
    .select("data_cotacao", "cotacao_compra", "sem_cotacao")
    .orderBy("data_cotacao")
)

(df_silver_cotacao.write
    .format("delta").mode("overwrite").option("overwriteSchema", True)
    .saveAsTable(f"{silver_schema}.tb_cotacao_dolar"))

display(df_silver_cotacao)